<a href="https://colab.research.google.com/github/par21val-512/smbpls/blob/main/smbpls_scvi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
! pip install anndata mudata

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 105.3 MB/s eta 0:00:00


In [3]:
import torch
from torch import nn
import numpy as np

import sklearn

from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.cross_decomposition import PLSRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from sklearn import datasets
import matplotlib.pyplot as plt

import warnings
import anndata as ad
import mudata as mu
import pandas as pd

RANDOM_SEED = 42

def generate_mudata(RANDOM_SEED):
    torch.manual_seed(RANDOM_SEED)

    n = 1000
    p1, p2, p3 = 50, 80, 30
    k = 2

    z = torch.normal(mean=0, std=1, size=(n, k))
    stdev_scaler = 0

    true_w3 = torch.randn(p1, k)
    true_w4 = torch.randn(p2, k)
    true_w5 = torch.randn(p3, k)

    x1 = z @ true_w3.T
    x2 = z @ true_w4.T
    x3 = z @ true_w5.T

    # Sparse latent
    z_sparse = z.clone()
    z_sparse[:500, 0] = 0
    z_sparse[500:, 1] = 0

    x1_sparse = z_sparse @ true_w3.T
    x2_sparse = z_sparse @ true_w4.T
    x3_sparse = z_sparse @ true_w5.T

    # Train/test split
    indices = np.arange(n)
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=RANDOM_SEED
    )

    split_labels = np.array(["train"] * n)
    split_labels[test_idx] = "test"

    # Helper to build AnnData
    def build_adata(X, X_sparse, name):
        adata = ad.AnnData(X=X.numpy())
        adata.layers["sparse"] = X_sparse.numpy()
        adata.obs["split"] = split_labels
        adata.var["feature"] = [f"{name}_{i}" for i in range(X.shape[1])]
        return adata

    adata1 = build_adata(x1, x1_sparse, "mod1")
    adata2 = build_adata(x2, x2_sparse, "mod2")
    adata3 = build_adata(x3, x3_sparse, "mod3")

    mdata = mu.MuData({
        "mod1": adata1,
        "mod2": adata2,
        "mod3": adata3,
    })

    return mdata

In [4]:
mdata = generate_mudata(RANDOM_SEED)

/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:571: UserWarning: Cannot join columns with the same name because var_names are intersecting.
  self._update_attr_legacy(attr, axis, join_common, **kwargs)
/usr/local/lib/python3.12/dist-packages/mudata/_core/mudata.py:1275: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility